In [1]:
# ═══ Cell 1: Install dependencies ═══
!pip install sentence-transformers faiss-cpu --quiet

import sentence_transformers
import faiss
print(f"sentence-transformers: {sentence_transformers.__version__}")
print(f"faiss-cpu:             {faiss.__version__ if hasattr(faiss, '__version__') else 'installed'}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 81.6 MB/s eta 0:00:00
sentence-transformers: 5.6.0
faiss-cpu:             1.15.0


In [2]:
# ═══ Cell 2: Mount Drive + Clone repo (if needed) + Load KB ═══
import sys, json
from pathlib import Path

# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

# --- Clone or pull repo ---
REPO_ROOT = Path("/content/slm-training")
if not REPO_ROOT.exists():
    !git clone https://github.com/trannhatbuilder/slm-training.git /content/slm-training
else:
    %cd /content/slm-training
    !git pull

KB_ROOT = REPO_ROOT / "data" / "kb"
sys.path.insert(0, str(REPO_ROOT / "src"))

# --- Load KB entries ---
from harness.kb.loader import KBLoader

loader = KBLoader(kb_root=str(KB_ROOT))
entries = loader.load_all()

print(f"KB entries loaded: {len(entries)}")
print(f"Load errors:       {len(loader.load_errors)}")

# --- Show category breakdown ---
from collections import Counter
cats = Counter(e.category for e in entries)
for cat, cnt in sorted(cats.items(), key=lambda x: -x[1]):
    print(f"  {cat:30s} {cnt}")

Mounted at /content/drive
Cloning into '/content/slm-training'...
remote: Enumerating objects: 192, done.
remote: Counting objects: 100% (192/192), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 192 (delta 58), reused 177 (delta 46), pack-reused 0 (from 0)
Receiving objects: 100% (192/192), 17.43 MiB | 25.98 MiB/s, done.
Resolving deltas: 100% (58/58), done.
KB entries loaded: 91
Load errors:       0
  sop                            19
  writing_guidelines             17
  remediation_guidance           12
  severity_guidance              7
  classification_criteria        6
  escalation_rules               6
  evidence_standards             6
  validation_requirements        6
  consistency_rules              4
  governance_rules               4
  taxonomy_definitions           4


In [3]:
# ═══ Cell 3: Chunk KB entries → Build embeddings → FAISS index ═══
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import time

# ── 3a. Chunks: mỗi KB entry → 1 chunk (title + description + conditions + action) ──
# KB entries đã nhỏ (ngắn), không cần chunk thêm. Tạo 1 text block cho mỗi entry.

chunks = []
for entry in entries:
    parts = [f"[{entry.kb_id}] {entry.title}"]
    if entry.description:
        parts.append(entry.description)
    if entry.conditions:
        parts.append("Conditions: " + "; ".join(entry.conditions))
    if entry.action:
        parts.append("Action: " + entry.action)
    chunk_text = "\n".join(parts)
    chunks.append({
        "kb_id": entry.kb_id,
        "category": entry.category,
        "text": chunk_text,
        "entry": entry,
    })

print(f"Total chunks: {len(chunks)}")
print(f"\n--- Sample chunk ---")
print(chunks[0]["text"][:300])

# ── 3b. Load embedding model ──
EMBED_MODEL = "all-MiniLM-L6-v2"  # 384 dims, fast, good for English
print(f"\nLoading embedding model: {EMBED_MODEL}...")
t0 = time.time()
embedder = SentenceTransformer(EMBED_MODEL)
print(f"Model loaded in {time.time()-t0:.1f}s")
print(f"Embedding dimension: {embedder.get_sentence_embedding_dimension()}")

# ── 3c. Encode all chunks ──
texts = [c["text"] for c in chunks]
print(f"\nEncoding {len(texts)} chunks...")
t0 = time.time()
embeddings = embedder.encode(texts, show_progress_bar=True, batch_size=32)
embeddings = np.array(embeddings, dtype="float32")
print(f"Encoded in {time.time()-t0:.1f}s")
print(f"Embeddings shape: {embeddings.shape}")

# ── 3d. Build FAISS index (Inner Product = cosine sau normalize) ──
faiss.normalize_L2(embeddings)  # cosine similarity = IP trên L2-normalized vectors
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)   # Inner Product index
index.add(embeddings)
print(f"\nFAISS index built: {index.ntotal} vectors, dim={dim}")

Total chunks: 91

--- Sample chunk ---
[KB-CLASS-001] Confirmed Vulnerability requires demonstrated exploitation + manual verification
To classify as Confirmed Vulnerability, three conditions must ALL be met: (1) exploitation is demonstrated in the evidence, (2) evidence supports the vulnerability claim, and (3) manual verification was p

Loading embedding model: all-MiniLM-L6-v2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded in 6.6s
Embedding dimension: 384

Encoding 91 chunks...


/tmp/ipykernel_857/1706229860.py:37: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {embedder.get_sentence_embedding_dimension()}")


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Encoded in 0.9s
Embeddings shape: (91, 384)

FAISS index built: 91 vectors, dim=384


In [4]:
# ═══ Cell 4: Semantic retrieval test — severity, validation, remediation, style ═══
from harness.kb.retriever import KBRetriever

retriever = KBRetriever(kb_root=str(KB_ROOT))

# ── Test queries: mô phỏng câu hỏi thực tế khi review finding ──
test_queries = [
    # --- Severity ---
    {
        "query_id": "SEM-SEV-001",
        "category": "severity",
        "text": "Finding has severity Critical with CVSS 9.8, what severity rules apply?",
        "expected_kb_ids": ["KB-SEV-001", "KB-SEV-006", "KB-CONS-001"],
    },
    {
        "query_id": "SEM-SEV-002",
        "category": "severity",
        "text": "Reported severity is High but CVSS score is 5.2, is this a mismatch?",
        "expected_kb_ids": ["KB-SEV-002", "KB-CONS-001"],
    },
    # --- Validation ---
    {
        "query_id": "SEM-VAL-001",
        "category": "validation",
        "text": "Check if finding has all required sections: observation, evidence, recommendation",
        "expected_kb_ids": ["KB-VAL-001", "KB-VAL-002", "KB-VAL-003"],
    },
    {
        "query_id": "SEM-VAL-002",
        "category": "validation",
        "text": "CVSS vector format validation and CWE identifier check",
        "expected_kb_ids": ["KB-VAL-002", "KB-VAL-003"],
    },
    # --- Remediation ---
    {
        "query_id": "SEM-REC-001",
        "category": "remediation",
        "text": "Hardcoded credentials in mobile app, what remediation template applies?",
        "expected_kb_ids": ["KB-REC-TPL-001", "KB-REC-TPL-006"],
    },
    {
        "query_id": "SEM-REC-002",
        "category": "remediation",
        "text": "Weak JWT algorithm HS256 needs migration to RS256",
        "expected_kb_ids": ["KB-REC-TPL-002"],
    },
    # --- Style ---
    {
        "query_id": "SEM-STYLE-001",
        "category": "style",
        "text": "Report heading hierarchy and metadata table format for pentest report",
        "expected_kb_ids": ["KB-STYLE-001", "KB-STYLE-004"],
    },
    {
        "query_id": "SEM-STYLE-002",
        "category": "style",
        "text": "Client asks about finding outside assessment scope, how to answer",
        "expected_kb_ids": ["KB-QA-005", "KB-QA-002"],
    },
]

# ── Run semantic search for each query ──
K = 5  # top-5
results = []

for q in test_queries:
    # Embed query
    q_emb = embedder.encode([q["text"]], normalize_embeddings=True)
    q_emb = np.array(q_emb, dtype="float32")

    # FAISS search
    scores, indices = index.search(q_emb, K)

    retrieved_ids = [chunks[idx]["kb_id"] for idx in indices[0]]
    retrieved_scores = [float(s) for s in scores[0]]

    # Check which expected IDs were found
    found = [eid for eid in q["expected_kb_ids"] if eid in retrieved_ids]
    missing = [eid for eid in q["expected_kb_ids"] if eid not in retrieved_ids]
    recall = len(found) / len(q["expected_kb_ids"]) if q["expected_kb_ids"] else 1.0

    result = {
        "query_id": q["query_id"],
        "category": q["category"],
        "query_text": q["text"],
        "top_k": K,
        "retrieved": list(zip(retrieved_ids, retrieved_scores)),
        "expected_kb_ids": q["expected_kb_ids"],
        "expected_found": found,
        "expected_missing": missing,
        "recall": recall,
    }
    results.append(result)

    status = "✓" if recall == 1.0 else "✗"
    print(f"{status} {q['query_id']} [{q['category']:12s}] recall={recall:.2f}  top5={retrieved_ids}")

# ── Summary ──
by_cat = {}
for r in results:
    cat = r["category"]
    by_cat.setdefault(cat, []).append(r["recall"])

print(f"\n{'Category':15s} {'Mean Recall':>12s} {'Queries':>8s}")
print("-" * 37)
for cat, recalls in sorted(by_cat.items()):
    print(f"{cat:15s} {np.mean(recalls):>12.2f} {len(recalls):>8d}")
overall = np.mean([r["recall"] for r in results])
print("-" * 37)
print(f"{'OVERALL':15s} {overall:>12.2f} {len(results):>8d}")

✗ SEM-SEV-001 [severity    ] recall=0.67  top5=['KB-CONS-001', 'KB-SEV-002', 'KB-SEV-001', 'KB-STYLE-004', 'KB-SEV-003']
✓ SEM-SEV-002 [severity    ] recall=1.00  top5=['KB-CONS-001', 'KB-SEV-002', 'KB-SEV-003', 'KB-SEV-004', 'KB-SEV-005']
✗ SEM-VAL-001 [validation  ] recall=0.33  top5=['KB-EVID-006', 'KB-STYLE-004', 'KB-VAL-001', 'KB-QA-003', 'KB-VAL-004']
✗ SEM-VAL-002 [validation  ] recall=0.50  top5=['KB-VAL-002', 'KB-CONS-001', 'KB-SEV-002', 'KB-SEV-003', 'KB-SEV-001']
✓ SEM-REC-001 [remediation ] recall=1.00  top5=['KB-REC-TPL-001', 'KB-REC-006', 'KB-REC-TPL-003', 'KB-REC-TPL-006', 'KB-REC-TPL-002']
✓ SEM-REC-002 [remediation ] recall=1.00  top5=['KB-REC-TPL-002', 'KB-CLASS-005', 'KB-CONS-004', 'KB-REC-004', 'KB-ESC-003']
✓ SEM-STYLE-001 [style       ] recall=1.00  top5=['KB-STYLE-001', 'KB-STYLE-005', 'KB-STYLE-004', 'KB-STYLE-006', 'KB-SCH-002']
✓ SEM-STYLE-002 [style       ] recall=1.00  top5=['KB-QA-005', 'KB-QA-003', 'KB-QA-001', 'KB-QA-002', 'KB-EVID-002']

Category        

In [5]:
# ═══ Cell 5: Hybrid retrieval + 3-method comparison ═══
import time

# ── Hybrid: merge metadata-filter results + semantic top-K, deduplicate, re-rank ──
def hybrid_retrieve(query_text, taxonomy_codes=None, domain=None, categories=None,
                    embedder=embedder, index=index, chunks=chunks,
                    retriever=retriever, semantic_k=5, meta_limit=20):
    """Combine metadata-filter + semantic retrieval, deduplicate, return merged."""

    # 1. Metadata-filter retrieval
    meta_result = retriever.retrieve_for_review(
        taxonomy_codes=taxonomy_codes,
        domain=domain if domain != "all" else None,
        categories=categories,
    )
    meta_ids = [e.kb_id for e in meta_result.entries[:meta_limit]]

    # 2. Semantic retrieval
    q_emb = embedder.encode([query_text], normalize_embeddings=True)
    q_emb = np.array(q_emb, dtype="float32")
    scores, indices = index.search(q_emb, semantic_k)
    sem_ids = [chunks[idx]["kb_id"] for idx in indices[0]]
    sem_scores = dict(zip(sem_ids, [float(s) for s in scores[0]]))

    # 3. Merge: semantic results first (scored), then metadata-only results (unscored)
    seen = set()
    merged = []
    # Semantic results (higher priority — relevance-scored)
    for kb_id in sem_ids:
        if kb_id not in seen:
            seen.add(kb_id)
            merged.append({"kb_id": kb_id, "source": "semantic", "score": sem_scores[kb_id]})
    # Metadata results (supplementary)
    for kb_id in meta_ids:
        if kb_id not in seen:
            seen.add(kb_id)
            merged.append({"kb_id": kb_id, "source": "metadata", "score": None})

    return {
        "merged": merged,
        "semantic_ids": sem_ids,
        "meta_ids": meta_ids,
    }

# ── Re-run all 8 queries with hybrid ──
hybrid_results = []

for q in test_queries:
    hybrid = hybrid_retrieve(
        query_text=q["text"],
        taxonomy_codes=None,  # intentionally no taxonomy hint — test pure hybrid
        domain="all",
    )

    merged_ids = [m["kb_id"] for m in hybrid["merged"]]
    found = [eid for eid in q["expected_kb_ids"] if eid in merged_ids]
    missing = [eid for eid in q["expected_kb_ids"] if eid not in merged_ids]
    recall = len(found) / len(q["expected_kb_ids"]) if q["expected_kb_ids"] else 1.0

    # Precision@5
    top5 = merged_ids[:5]
    relevant = set(q["expected_kb_ids"])
    prec5 = sum(1 for mid in top5 if mid in relevant) / 5

    result = {
        "query_id": q["query_id"],
        "category": q["category"],
        "recall_hybrid": recall,
        "precision5_hybrid": prec5,
        "merged_ids": merged_ids[:10],  # top 10
        "expected_found": found,
        "expected_missing": missing,
    }
    hybrid_results.append(result)

    status = "✓" if recall == 1.0 else "✗"
    print(f"{status} {q['query_id']:15s} recall={recall:.2f}  prec@5={prec5:.2f}  top5={top5}")

# ── Summary comparison ──
print(f"\n{'Category':15s} {'Semantic':>10s} {'Hybrid':>10s}")
print("-" * 37)
for cat in ["severity", "validation", "remediation", "style"]:
    sem_recalls = [r["recall"] for r in results if r["category"] == cat]
    hyb_recalls = [r["recall_hybrid"] for r in hybrid_results if r["category"] == cat]
    print(f"{cat:15s} {np.mean(sem_recalls):>10.2f} {np.mean(hyb_recalls):>10.2f}")

sem_overall = np.mean([r["recall"] for r in results])
hyb_overall = np.mean([r["recall_hybrid"] for r in hybrid_results])
print("-" * 37)
print(f"{'OVERALL':15s} {sem_overall:>10.2f} {hyb_overall:>10.2f}")

✗ SEM-SEV-001     recall=0.67  prec@5=0.40  top5=['KB-CONS-001', 'KB-SEV-002', 'KB-SEV-001', 'KB-STYLE-004', 'KB-SEV-003']
✓ SEM-SEV-002     recall=1.00  prec@5=0.40  top5=['KB-CONS-001', 'KB-SEV-002', 'KB-SEV-003', 'KB-SEV-004', 'KB-SEV-005']
✗ SEM-VAL-001     recall=0.33  prec@5=0.20  top5=['KB-EVID-006', 'KB-STYLE-004', 'KB-VAL-001', 'KB-QA-003', 'KB-VAL-004']
✗ SEM-VAL-002     recall=0.50  prec@5=0.20  top5=['KB-VAL-002', 'KB-CONS-001', 'KB-SEV-002', 'KB-SEV-003', 'KB-SEV-001']
✓ SEM-REC-001     recall=1.00  prec@5=0.40  top5=['KB-REC-TPL-001', 'KB-REC-006', 'KB-REC-TPL-003', 'KB-REC-TPL-006', 'KB-REC-TPL-002']
✓ SEM-REC-002     recall=1.00  prec@5=0.20  top5=['KB-REC-TPL-002', 'KB-CLASS-005', 'KB-CONS-004', 'KB-REC-004', 'KB-ESC-003']
✓ SEM-STYLE-001   recall=1.00  prec@5=0.40  top5=['KB-STYLE-001', 'KB-STYLE-005', 'KB-STYLE-004', 'KB-STYLE-006', 'KB-SCH-002']
✓ SEM-STYLE-002   recall=1.00  prec@5=0.40  top5=['KB-QA-005', 'KB-QA-003', 'KB-QA-001', 'KB-QA-002', 'KB-EVID-002']

Cate

In [6]:
# ═══ Cell 6: Full comparison — Metadata-only vs Semantic-only vs Hybrid (with taxonomy) ═══
# Dùng lại 10 gold queries từ retrieval_eval.json

with open(KB_ROOT / "retrieval_eval.json", "r") as f:
    existing_eval = json.load(f)

# ── Run 3 methods on each of the 10 gold queries ──
comparison = []

for q in existing_eval["queries"]:
    qid = q["query_id"]
    params = q["query_params"]
    expected = q["expected_kb_ids"]
    relevant = q.get("relevant_kb_ids", expected)

    taxonomy_codes = params.get("taxonomy_codes")
    domain = params.get("domain")
    categories = params.get("categories")
    query_text = q["scenario"]  # use scenario as semantic query

    # --- Method 1: Metadata-filter only ---
    meta_result = retriever.retrieve_for_review(
        taxonomy_codes=taxonomy_codes,
        domain=domain if domain != "all" else None,
        categories=categories,
    )
    meta_ids = [e.kb_id for e in meta_result.entries]
    meta_found = [e for e in expected if e in meta_ids]
    meta_recall = len(meta_found) / len(expected) if expected else 1.0
    meta_prec5 = sum(1 for mid in meta_ids[:5] if mid in relevant) / 5

    # --- Method 2: Semantic-only ---
    q_emb = embedder.encode([query_text], normalize_embeddings=True)
    q_emb = np.array(q_emb, dtype="float32")
    scores, indices = index.search(q_emb, 5)
    sem_ids = [chunks[idx]["kb_id"] for idx in indices[0]]
    sem_found = [e for e in expected if e in sem_ids]
    sem_recall = len(sem_found) / len(expected) if expected else 1.0
    sem_prec5 = sum(1 for mid in sem_ids if mid in relevant) / 5

    # --- Method 3: Hybrid (metadata + semantic, dedup, semantic first) ---
    merged_ids = []
    seen = set()
    for mid in sem_ids:
        if mid not in seen:
            seen.add(mid)
            merged_ids.append(mid)
    for mid in meta_ids:
        if mid not in seen:
            seen.add(mid)
            merged_ids.append(mid)
    hyb_found = [e for e in expected if e in merged_ids]
    hyb_recall = len(hyb_found) / len(expected) if expected else 1.0
    hyb_prec5 = sum(1 for mid in merged_ids[:5] if mid in relevant) / 5

    comparison.append({
        "query_id": qid,
        "meta_recall": meta_recall, "meta_prec5": meta_prec5,
        "sem_recall": sem_recall,   "sem_prec5": sem_prec5,
        "hyb_recall": hyb_recall,   "hyb_prec5": hyb_prec5,
    })

    print(f"{qid}  meta={meta_recall:.0%}/{meta_prec5:.0%}  sem={sem_recall:.0%}/{sem_prec5:.0%}  hyb={hyb_recall:.0%}/{hyb_prec5:.0%}")

# ── Aggregate ──
def agg(key): return np.mean([c[key] for c in comparison])

print(f"\n{'Method':20s} {'Mean Recall':>12s} {'Mean Prec@5':>12s}")
print("-" * 46)
print(f"{'Metadata-filter':20s} {agg('meta_recall'):>12.2f} {agg('meta_prec5'):>12.2f}")
print(f"{'Semantic-only':20s} {agg('sem_recall'):>12.2f} {agg('sem_prec5'):>12.2f}")
print(f"{'Hybrid':20s} {agg('hyb_recall'):>12.2f} {agg('hyb_prec5'):>12.2f}")

Q001  meta=100%/60%  sem=100%/40%  hyb=100%/40%
Q002  meta=100%/40%  sem=100%/60%  hyb=100%/60%
Q003  meta=100%/20%  sem=100%/60%  hyb=100%/60%
Q004  meta=100%/20%  sem=100%/20%  hyb=100%/20%
Q005  meta=100%/0%  sem=100%/40%  hyb=100%/40%
Q006  meta=100%/40%  sem=50%/60%  hyb=100%/60%
Q007  meta=100%/60%  sem=100%/60%  hyb=100%/60%
Q008  meta=100%/20%  sem=100%/60%  hyb=100%/60%
Q009  meta=100%/60%  sem=50%/60%  hyb=100%/60%
Q010  meta=100%/40%  sem=0%/0%  hyb=100%/0%

Method                Mean Recall  Mean Prec@5
----------------------------------------------
Metadata-filter              1.00         0.36
Semantic-only                0.80         0.46
Hybrid                       1.00         0.46


In [7]:
# ═══ Cell 7: Save index, embeddings, and updated retrieval_eval.json ═══
import shutil
from datetime import datetime, timezone

# ── 7a. Save FAISS index + embeddings to repo ──
INDEX_DIR = REPO_ROOT / "data" / "kb" / "index"
INDEX_DIR.mkdir(exist_ok=True)

faiss.write_index(index, str(INDEX_DIR / "kb_faiss.index"))
np.save(str(INDEX_DIR / "kb_embeddings.npy"), embeddings)

# Save chunk metadata (kb_id + category mapping)
chunk_meta = [{"kb_id": c["kb_id"], "category": c["category"]} for c in chunks]
with open(INDEX_DIR / "chunk_metadata.json", "w") as f:
    json.dump(chunk_meta, f, indent=2)

print(f"FAISS index saved:  {INDEX_DIR / 'kb_faiss.index'}")
print(f"Embeddings saved:   {INDEX_DIR / 'kb_embeddings.npy'}")
print(f"Chunk metadata:     {INDEX_DIR / 'chunk_metadata.json'}")

# ── 7b. Build updated retrieval_eval.json with hybrid results ──
updated_queries = []
for q in existing_eval["queries"]:
    qid = q["query_id"]
    params = q["query_params"]
    expected = q["expected_kb_ids"]
    relevant = q.get("relevant_kb_ids", expected)
    query_text = q["scenario"]
    taxonomy_codes = params.get("taxonomy_codes")
    domain = params.get("domain")
    categories = params.get("categories")

    # Re-run hybrid
    meta_result = retriever.retrieve_for_review(
        taxonomy_codes=taxonomy_codes,
        domain=domain if domain != "all" else None,
        categories=categories,
    )
    meta_ids = [e.kb_id for e in meta_result.entries]

    q_emb = embedder.encode([query_text], normalize_embeddings=True)
    q_emb = np.array(q_emb, dtype="float32")
    scores, indices = index.search(q_emb, 5)
    sem_ids = [chunks[idx]["kb_id"] for idx in indices[0]]
    sem_scores = [float(s) for s in scores[0]]

    # Merge hybrid
    merged = []
    seen = set()
    for mid, sc in zip(sem_ids, sem_scores):
        if mid not in seen:
            seen.add(mid)
            merged.append(mid)
    for mid in meta_ids:
        if mid not in seen:
            seen.add(mid)
            merged.append(mid)

    found = [e for e in expected if e in merged]
    missing = [e for e in expected if e not in merged]
    recall = len(found) / len(expected) if expected else 1.0
    prec5 = sum(1 for mid in merged[:5] if mid in relevant) / 5

    updated_queries.append({
        "query_id": qid,
        "scenario": q["scenario"],
        "query_params": params,
        "retrieval_method": "hybrid (semantic top-5 + metadata-filter)",
        "semantic_top5": list(zip(sem_ids, sem_scores)),
        "metadata_ids": meta_ids,
        "hybrid_merged": merged[:15],
        "expected_kb_ids": expected,
        "relevant_kb_ids": relevant,
        "expected_found": found,
        "expected_missing": missing,
        "recall": recall,
        "precision_at_5": prec5,
    })

# ── 7c. Write retrieval_eval.json ──
eval_out = {
    "eval_version": "2.0",
    "kb_version": "1.1",
    "evaluated_at": datetime.now(timezone.utc).isoformat(),
    "retrieval_methods_compared": {
        "metadata_filter": {"mean_recall": 1.0, "mean_precision_at_5": 0.36},
        "semantic_only":   {"mean_recall": 0.80, "mean_precision_at_5": 0.46},
        "hybrid":          {"mean_recall": 1.0,  "mean_precision_at_5": 0.46},
    },
    "selected_method": "hybrid",
    "embedding_model": EMBED_MODEL,
    "embedding_dim": 384,
    "faiss_index_type": "IndexFlatIP (cosine, L2-normalized)",
    "total_queries": 10,
    "aggregate_metrics": {
        "mean_recall": 1.0,
        "mean_precision_at_5": 0.46,
        "pass_rate": 1.0,
        "queries_passed": 10,
        "queries_failed": 0,
    },
    "coverage_categories": existing_eval["coverage_categories"],
    "queries": updated_queries,
}

eval_path = KB_ROOT / "retrieval_eval.json"
with open(eval_path, "w") as f:
    json.dump(eval_out, f, indent=2)

print(f"\nretrieval_eval.json updated: {eval_path}")
print(f"  eval_version:    2.0")
print(f"  selected_method: hybrid")
print(f"  mean_recall:     1.00")
print(f"  mean_prec@5:     0.46")

FAISS index saved:  /content/slm-training/data/kb/index/kb_faiss.index
Embeddings saved:   /content/slm-training/data/kb/index/kb_embeddings.npy
Chunk metadata:     /content/slm-training/data/kb/index/chunk_metadata.json

retrieval_eval.json updated: /content/slm-training/data/kb/retrieval_eval.json
  eval_version:    2.0
  selected_method: hybrid
  mean_recall:     1.00
  mean_prec@5:     0.46


In [8]:
# ═══ Cell 8: Persistent backup + Git commit ═══
import shutil

# ── 8a. Backup to Google Drive ──
DRIVE_KB_DIR = Path("/content/drive/MyDrive/evvo-slm-checkpoints/kb-index-v1")
DRIVE_KB_DIR.mkdir(parents=True, exist_ok=True)

for fname in ["kb_faiss.index", "kb_embeddings.npy", "chunk_metadata.json"]:
    src = INDEX_DIR / fname
    dst = DRIVE_KB_DIR / fname
    shutil.copy2(src, dst)
    print(f"  Copied: {fname}")

# Also backup retrieval_eval.json
shutil.copy2(KB_ROOT / "retrieval_eval.json", DRIVE_KB_DIR / "retrieval_eval.json")
print(f"  Copied: retrieval_eval.json")

print(f"\nGoogle Drive backup: {DRIVE_KB_DIR}")

# ── 8b. Git commit & push ──
%cd /content/slm-training

!git add data/kb/index/ data/kb/retrieval_eval.json
!git status

  Copied: kb_faiss.index
  Copied: kb_embeddings.npy
  Copied: chunk_metadata.json
  Copied: retrieval_eval.json

Google Drive backup: /content/drive/MyDrive/evvo-slm-checkpoints/kb-index-v1
/content/slm-training
On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   data/kb/index/chunk_metadata.json
	new file:   data/kb/index/kb_embeddings.npy
	new file:   data/kb/index/kb_faiss.index
	modified:   data/kb/retrieval_eval.json



In [10]:
# ═══ Cell 9: Git commit & push ═══
%cd /content/slm-training

!git commit -m "feat(retrieval): add FAISS vector index + hybrid retrieval eval v2.0"

!git push origin main

/content/slm-training
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@ca658421bc10.(none)')
fatal: could not read Username for 'https://github.com': No such device or address
